# Silver Layer

## Data Standardization

Purpose:
- Read Bronze dataset
- Apply Silver transformations
- Persist standardized data
- Validate results

## Environment Bootstrap

Required on Databricks Free Edition: there is no `libraries: - whl: ...` mechanism wired to a classic/serverless cluster here, so the project wheel has to be pip-installed explicitly in the notebook before any `data_platform`/`integrations` import works.

`wheel_path` comes from a Job base_parameter (`${workspace.root_path}/artifacts/.internal`, resolved by the Databricks bundle at deploy time) -- portable across users and targets (dev/prod), unlike a hardcoded `/Workspace/Users/<you>/...` path. Bundle substitutions only expand inside bundle YAML files, never inside notebook content directly, which is why this goes through a widget instead of being inlined here.

In [ ]:
dbutils.widgets.text("wheel_path", "")
wheel_path = dbutils.widgets.get("wheel_path")
wheel_glob = f"{wheel_path}/*.whl"

%pip install $wheel_glob

dbutils.library.restartPython()

## Imports

In [ ]:
from data_platform.compute.delta_io import read_delta, write_delta
from data_platform.compute.spark import get_spark
from data_platform.storage.config import StorageConfig
from integrations.databricks.runtime.parameters import get_parameter

from data_platform.processing.silver.transformations import (
    apply_standard_transformations,
)

## Parameters

In [ ]:
entity = get_parameter("entity")

## Spark Session

In [ ]:
spark = get_spark("Silver Layer")

## Read Bronze

In [ ]:
bronze_path = StorageConfig.bronze(entity)
silver_path = StorageConfig.silver(entity)

df = read_delta(spark=spark, path=bronze_path)

## Transform Data

In [ ]:
df = apply_standard_transformations(df)

## Write Silver

In [ ]:
write_delta(df=df, path=silver_path, mode="overwrite")

## Validation

In [ ]:
print(f"Rows: {df.count()}")

df.printSchema()

display(df.limit(10))